# 🌿 PixelRotina — Compilador de APK no Google Colab

Este notebook configura automaticamente **OpenJDK 21**, **Gradle 9.3.1** e **Android SDK (API 36 / 36.1)** no Google Colab para executar os testes Robolectric e gerar os arquivos **`PixelRotina-debug.apk`** e **`PixelRotina-release.apk`** prontos para instalar no seu Android.

### Como usar:
1. Escolha na **Etapa 1** como carregar o projeto: **GitHub**, **Upload de `.zip`** (exportado do AI Studio) ou **Google Drive**.
2. Execute as células em ordem (**Runtime → Run all** ou clicando no botão ▶ de cada célula).
3. Na **Etapa 4**, baixe o APK direto para o seu computador/celular ou salve uma cópia no seu Google Drive.

## 📦 Etapa 1 — Carregar o Código do Projeto
Escolha uma das 3 origens abaixo (`github`, `zip_upload` ou `google_drive`).

In [ ]:
#@title Configuração da Origem do Projeto
ORIGEM = "github" #@param ["github", "zip_upload", "google_drive"]
GITHUB_REPO_URL = "https://github.com/SEU_USUARIO/PixelRotina.git" #@param {type:"string"}
GITHUB_BRANCH = "main" #@param {type:"string"}
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/PixelRotina" #@param {type:"string"}

import os, shutil, zipfile

PROJECT_DIR = "/content/PixelRotina"

if ORIGEM == "github":
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    !git clone --depth 1 -b {GITHUB_BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}

elif ORIGEM == "zip_upload":
    from google.colab import files
    print("Selecione o arquivo .zip do projeto PixelRotina...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded.keys()))
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.makedirs(PROJECT_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall(PROJECT_DIR)
    # Se o ZIP tiver uma pasta raiz interna, ajusta PROJECT_DIR automaticamente
    entries = os.listdir(PROJECT_DIR)
    if len(entries) == 1 and os.path.isdir(os.path.join(PROJECT_DIR, entries[0])):
        inner = os.path.join(PROJECT_DIR, entries[0])
        if os.path.exists(os.path.join(inner, "settings.gradle.kts")):
            PROJECT_DIR = inner

elif ORIGEM == "google_drive":
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(DRIVE_PROJECT_PATH, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"✅ Projeto carregado em: {os.getcwd()}")
!ls -la

## 🛠️ Etapa 2 — Configurar Ambiente (JDK 21 + Gradle 9.3.1 + Android SDK 36.1) e Compilar
Executa automaticamente o script `scripts/build_colab.sh`. Na primeira execução o download do JDK + SDK leva cerca de 2 a 4 minutos.

In [ ]:
#@title Opções de Compilação
BUILD_TYPE = "both" #@param ["both", "debug", "release"]
EXECUTAR_TESTES_ROBOLECTRIC = True #@param {type:"boolean"}
RUN_TESTS_STR = "true" if EXECUTAR_TESTES_ROBOLECTRIC else "false"

!chmod +x scripts/build_colab.sh
!bash scripts/build_colab.sh {BUILD_TYPE} {RUN_TESTS_STR}

## 📲 Etapa 3 — Baixar o APK Compilado
Escolha qual APK baixar diretamente para o seu computador ou celular.

In [ ]:
#@title Download do APK
APK_PARA_BAIXAR = "PixelRotina-release.apk" #@param ["PixelRotina-release.apk", "PixelRotina-debug.apk"]

import os
from google.colab import files

apk_path = os.path.join(os.getcwd(), "dist", APK_PARA_BAIXAR)
if os.path.exists(apk_path):
    print(f"📥 Iniciando download de {apk_path}...")
    files.download(apk_path)
else:
    print(f"⚠️ Arquivo não encontrado: {apk_path}. Verifique se a Etapa 2 terminou com sucesso.")

## ☁️ Etapa 4 (Opcional) — Salvar APKs no Google Drive
Execute esta célula caso queira copiar os APKs gerados diretamente para a pasta `PixelRotina_APKs` no seu Google Drive.

In [ ]:
import os, shutil
from google.colab import drive

drive.mount('/content/drive')
dest_dir = "/content/drive/MyDrive/PixelRotina_APKs"
os.makedirs(dest_dir, exist_ok=True)

dist_dir = os.path.join(os.getcwd(), "dist")
for fname in os.listdir(dist_dir):
    if fname.endswith(".apk"):
        shutil.copy2(os.path.join(dist_dir, fname), os.path.join(dest_dir, fname))
        print(f"✅ Copiado para o Drive: {dest_dir}/{fname}")